# Calculate the SFS_f in Supplementary method for given Roulette mutation rate distributions, selection coefficients and ancestries

In [1]:
# --- make parent folder importable ---
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))  # points to Paper_SFS/

# --- usual imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc
from scipy.integrate import quad
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from scipy.stats import gmean, linregress, norm, beta, uniform, lognorm
from scipy.integrate import simpson, trapezoid
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.integrate import simpson
from scipy import stats
import glob
import os
import math
import dask
import dask.dataframe as dd
import re
import dask.bag as db
import csv
from tqdm import tqdm
from dask.diagnostics import ProgressBar
from concurrent.futures import ProcessPoolExecutor
import time
import joblib
from joblib import Parallel, delayed, parallel_backend
import multiprocessing
import pickle
import random
import json

# --- import your project modules (files in Paper_SFS/) ---
import Gen_SFS_with_s_demography
import plot_SFS

# (optional) pull specific functions
# from Gen_SFS_with_s import calc_SFS_k
# from plot_SFS import plot_sfs


In [ ]:
# Load demography and mutation rate parameters for a given demography and epoch
def load_mu_params_for_demography(demography, epoch):

    if demography == "NFE" or demography == "schraiber_et_al":
        df_mu_var_p = pd.read_csv('Mutation_rate_estimation/param_mut_rate/results_mu_var_p_NFE.txt', sep = '\t')
    elif demography == "all":
        df_mu_var_p = pd.read_csv('Mutation_rate_estimation/param_mut_rate/results_mu_var_p_all.txt', sep = '\t')
    else:
        df_mu_var_p = pd.read_csv(f'Mutation_rate_estimation/param_mut_rate/results_mu_var_p_{demography}_epoch_{epoch}.txt', sep = '\t')
         
    return df_mu_var_p
    

def load_demography_result(json_file, demography, epoch_no):
    with open(json_file, "r") as f:
        records = json.load(f)
    for rec in records:
        if rec["demography"] == demography and rec["epoch_no"] == epoch_no:
            return rec

    return {"params" : []}


In [ ]:
# Obtain the predicted SFS for a given demography, mutation rate, selection, and sample size n
def get_SFS_on_ancestry(demography, epoch, n, mu_low_mu = 2e-10, json_file = "Demography_estimation/demography_results.json", pkl_folder = "/n/data2/hms/dbmi/sunyaev/lab/pkar/demography_SFS/"):

    df_mu_var_p = load_mu_params_for_demography(demography = demography, epoch = epoch)
    MR = df_mu_var_p['MR'].values
    mean_mu_values = df_mu_var_p['mean_mu'].values
    var_mu_values = df_mu_var_p['var_mu'].values
    p_mu_values = df_mu_var_p['p_mu'].values

    slope, intercept, r_value, p_value, std_err = linregress(MR[:-1], mean_mu_values[:-1]/(4*14448))

    if demography == "all":
        mean_mus = np.insert(mean_mu_values, 0, (slope*0.004)*4*14448)
    else:
        mean_mus = np.insert(mean_mu_values, 0, (slope*0.004 + intercept)*4*14448)
    var_mus = np.insert(var_mu_values, 0, 1e-23*(4*14448)**2)
    p_mus = np.insert(p_mu_values, 0, 0)

    mu_var_pairs = np.column_stack((mean_mus, var_mus))
    s_values = -np.logspace(np.log10(10**-6), np.log10(100), num=500)  # 500 negative values
    s_values = np.append(s_values, 0)  # Include neutral case (total 501 values)

    # Load demography parameters
    params_rec = load_demography_result(json_file, demography, epoch)

    st_time = time.time()
    SFS_eachp = Gen_SFS_with_s_demography.compute_SFS_gamma_mu_var_pairs(mu_var_pairs, s_values, demography = demography, n = n, 
                                                                         params = params_rec["params"], model = "original", kmax = 5000)  
    
    high_mu_pkl_filename = pkl_folder+f"SFS_gamma_{demography}.pkl"
    with open(high_mu_pkl_filename, "wb") as f:
        pickle.dump(SFS_eachp, f)
    en_time = time.time()
    print(f"Done in {en_time-st_time}s")

    results = Parallel(n_jobs=-1)(
        delayed(Gen_SFS_with_s_demography.compute_SFS)(mu_low_mu, s, demography = demography, n=n, params = params_rec["params"], model = "original", kmax = 5000) for s in s_values
    )
    
    # Reshape results into (s_count, 5001)
    s_count = len(s_values)
    SFS_values = np.array([sfs for _, _, _, sfs in results])    
    SFS_matrix = SFS_values.reshape(s_count, -1)
    print(SFS_matrix.shape)

    low_mu_pkl_filename = pkl_folder+f"SFS_low_{demography}.pkl"
    with open(low_mu_pkl_filename, "wb") as f:
        pickle.dump(SFS_matrix, f)
    print("Done")

    p_mus_broadcasted = p_mus[:, np.newaxis, np.newaxis]
    SFS_low_mu = SFS_matrix[np.newaxis, :, :]
    SFS_each = (1-p_mus_broadcasted)*SFS_eachp + p_mus_broadcasted*SFS_low_mu

    pkl_filename = pkl_folder+f"SFS_{demography}.pkl"
    with open(pkl_filename, "wb") as f:
        pickle.dump(SFS_each, f)
    print("Done") 

In [4]:
# NFE
get_SFS_on_ancestry("NFE", epoch = 3, n = 1112012)

Done in 2777.433429956436s
(501, 5001)
Done
Done


In [4]:
# AFR
get_SFS_on_ancestry("AFR", epoch = 3, n = 33480)

Done in 3102.3088409900665s
(501, 5001)
Done
Done


In [4]:
# AMR
get_SFS_on_ancestry("AMR", epoch = 3, n = 44724)

Done in 16096.536698102951s
(501, 5001)
Done
Done


In [5]:
# EAS
get_SFS_on_ancestry("EAS", epoch = 3, n = 39700)

Done in 3004.6118273735046s
(501, 5001)
Done
Done


In [6]:
# FIN
get_SFS_on_ancestry("FIN", epoch = 3, n = 53420)

Done in 2810.121208190918s
(501, 5001)
Done
Done


In [7]:
# SAS
get_SFS_on_ancestry("SAS", epoch = 3, n = 86258)

Done in 2980.2059648036957s
(501, 5001)
Done
Done


In [4]:
# all
get_SFS_on_ancestry("all", epoch = 3, n = 1461892)

Done in 2765.498701572418s
(501, 5001)
Done
Done
